# Creating a Parquet for the Games Repository!
We share this notebook if you would like to see how we got to the creation of the parquet for the categories and genres of each game.

Due to file size limits, the dataset is hosted on Google Drive (and we will accordingly manage this in the future as we eventually lose proper google drive access to @ucsd.edu:

Link to the original steam games for categories and genres
https://www.kaggle.com/datasets/artermiloff/steam-games-dataset?select=games_march2025_full.cs

Download here:

(V1.0)
https://drive.google.com/uc?id=1VvelASY-o9ubjBcSZoUI9sseflzUUoDF

(V2.0)
https://drive.google.com/uc?id=1Mhu3IIGb9v4H7ojFEEadQ8xg07Dw-Aan

Place the file in an accessible path i.e.:
data/steam_tfidf_nn_recommender.parquet


# UPDATE for 2.0:
We incorporated release_date into the dataset to strengthen both exploratory analysis and future modeling. Time is a meaningful factor in the gaming market because player behavior, competition, and retention dynamics often differ by generation of games. Older titles may have established communities and long-tail engagement, while newer releases can experience launch spikes, rapid adoption, or short-term churn. By preserving release dates, we can organize competitor comparisons within the correct market era, measure lifecycle effects, and avoid treating a 2012 title the same as a 2025 release. This addition improves the interpretability of our EDA and creates a stronger foundation for any later predictive features related to game age, seasonality, or market timing.

# PART 1: Getting the data together
All the code here leading into part 2 is for the creation of the parquet, the next section will highlight quick use cases of what we can do with it.

In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "games_march2025_full.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "artermiloff/steam-games-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

#print("First 5 records:", dfa.head())

/tmp/ipykernel_17099/341907953.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 450M/450M [00:06<00:00, 68.5MB/s]


In [ ]:
import ast
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# --------------------------------------------------
# 1. Safe parser
# --------------------------------------------------
def parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x == '':
            return []
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, list) else []
        except:
            return []
    return []

def parse_tags(x):
    if pd.isna(x):
        return []
    if isinstance(x, dict):
        return list(x.keys())
    if isinstance(x, str):
        x = x.strip()
        if x == '':
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, dict):
                return list(parsed.keys())
        except:
            return []
    return []

# --------------------------------------------------
# 2. Keep only relevant columns
# --------------------------------------------------
cols_to_keep = [
    'appid',
    'name',
    'release_date', #v2.0 schema addition for temporal analysis
    'price',
    'genres',
    'tags',
    'peak_ccu',
    'estimated_owners',
    'average_playtime_forever',
    'pct_pos_total',
    'num_reviews_total'
]

df_rec = df[cols_to_keep].copy()

# convert release_date into datetime
df_rec['release_date'] = pd.to_datetime(
    df_rec['release_date'],
    errors='coerce'
)

# --------------------------------------------------
# 3. Parse list-like columns
# --------------------------------------------------
df_rec['genres_list'] = df_rec['genres'].apply(parse_list)
df_rec['tags_list'] = df_rec['tags'].apply(parse_tags)

# --------------------------------------------------
# 4. Build feature text for TF-IDF
#    tags = primary signal
#    genres = supporting signal
# --------------------------------------------------
df_rec['feature_tokens'] = df_rec['tags_list'] + df_rec['genres_list']

df_rec['feature_text'] = df_rec['feature_tokens'].apply(
    lambda x: ' | '.join([str(token).strip() for token in x if str(token).strip() != ''])
)

df_rec['feature_text'] = df_rec['feature_text'].replace('', 'unknown')

# --------------------------------------------------
# 5. TF-IDF vectorization
# --------------------------------------------------
vectorizer = TfidfVectorizer(
    token_pattern=r'[^|]+',
    lowercase=True,
    dtype=np.float32
)

X = vectorizer.fit_transform(df_rec['feature_text'])

print('TF-IDF matrix shape:', X.shape)
print('Number of TF-IDF features:', len(vectorizer.get_feature_names_out()))

# --------------------------------------------------
# NN + Competitor Building (UPDATED)
# 60 neighbors → top 50 competitors
# --------------------------------------------------

TOP_K = 50
NEIGHBOR_BUFFER = 10   # gives us room after removing self

nn_model = NearestNeighbors(
    n_neighbors=TOP_K + NEIGHBOR_BUFFER,   # 60
    metric='cosine',
    algorithm='brute',
    n_jobs=-1
)

nn_model.fit(X)

distances, indices = nn_model.kneighbors(X)

game_names = df_rec['name'].fillna('Unknown Game').tolist()
appids = df_rec['appid'].tolist()
#v2.0 schema addition for temporal analysis
release_dates = df_rec['release_date'].tolist()

top_50_competitors = []

for i in range(len(df_rec)):
    # pair up distances and indices
    candidate_pairs = list(zip(distances[i], indices[i]))

    # explicitly remove self
    candidate_pairs = [(dist, j) for dist, j in candidate_pairs if j != i]

    # keep top 50 after removal
    candidate_pairs = candidate_pairs[:TOP_K]

    row = []
    for dist, j in candidate_pairs:
        sim = 1.0 - float(dist)
        row.append({
          'appid': appids[j],
          'name': game_names[j],
          'release_date': (
              release_dates[j].strftime('%Y-%m-%d')
              if pd.notna(release_dates[j])
                else None
          ),
          'similarity': round(sim, 4)
      })

    top_50_competitors.append(row)

    if i % 5000 == 0 and i > 0:
        print(f'Built competitors for {i} / {len(df_rec)} games')

df_rec['top_50_competitors'] = top_50_competitors
df_rec['top_50_competitors_json'] = df_rec['top_50_competitors'].apply(json.dumps)

# --------------------------------------------------
# 8. Save-ready dataframe
# --------------------------------------------------
df_save = df_rec[
    [
        'appid',
        'name',
        'release_date', #v2.0 schema addition for temporal analysis
        'price',
        'genres',
        'tags',                   # categories repackaged
        'peak_ccu',
        'estimated_owners',
        'average_playtime_forever',
        'pct_pos_total',
        'num_reviews_total',
        'top_50_competitors_json'
    ]
].copy()

print('Done building recommender dataset.')
print(df_save[['appid', 'name','release_date','top_50_competitors_json']].head(3))

TF-IDF matrix shape: (94948, 976)
Number of TF-IDF features: 976
Built competitors for 5000 / 94948 games
Built competitors for 10000 / 94948 games
Built competitors for 15000 / 94948 games
Built competitors for 20000 / 94948 games
Built competitors for 25000 / 94948 games
Built competitors for 30000 / 94948 games
Built competitors for 35000 / 94948 games
Built competitors for 40000 / 94948 games
Built competitors for 45000 / 94948 games
Built competitors for 50000 / 94948 games
Built competitors for 55000 / 94948 games
Built competitors for 60000 / 94948 games
Built competitors for 65000 / 94948 games
Built competitors for 70000 / 94948 games
Built competitors for 75000 / 94948 games
Built competitors for 80000 / 94948 games
Built competitors for 85000 / 94948 games
Built competitors for 90000 / 94948 games
Done building recommender dataset.
    appid                 name release_date  \
0     730     Counter-Strike 2   2012-08-21   
1  578080  PUBG: BATTLEGROUNDS   2017-12-21   
2   

In [ ]:
# --------------------------------------------------
# 10. Save to parquet
# --------------------------------------------------
# output_path = '/content/drive/MyDrive/DSC 288 DS Grad Capstone/steam_tfidf_nn_recommender.parquet'
# df_save.to_parquet(output_path, index=False)

# print(f'Saved parquet to: {output_path}')

output_path = '/content/drive/MyDrive/DSC 288 DS Grad Capstone/steam_tfidf_nn_recommender_v2.parquet'
df_save.to_parquet(output_path, index=False)

print(f'Saved parquet to: {output_path}')

# --------------------------------------------------
# 11. Helper to inspect one game's competitors
# --------------------------------------------------
def show_competitors(df_in, game_name, top_n=50, sort_by='similarity'):
    row = df_in[df_in['name'].str.lower() == game_name.lower()]

    if row.empty:
        print(f'Game "{game_name}" not found.')
        return None

    comps = row.iloc[0]['top_50_competitors']

    if isinstance(comps, str):
        comps = json.loads(comps)

    comps_df = pd.DataFrame(comps)

    if sort_by == 'release_date':
        comps_df['release_date'] = pd.to_datetime(comps_df['release_date'])
        comps_df = comps_df.sort_values('release_date', ascending=True)

    else:
        comps_df = comps_df.sort_values('similarity', ascending=False)

    print(f'\nTop {top_n} competitors for: {row.iloc[0]["name"]}\n')
    return comps_df.head(top_n).reset_index(drop=True)

# --------------------------------------------------
# 12. Example usage
# --------------------------------------------------
# df_out = show_competitors(df_rec, 'Dota 2')
# display(df_out)

## OUTDATED: go to V2.0
# PART 2(V1.0): Here is a quick showcase of our use of this parquet for DOTA 2
You can begin running code for use cases here.

In [ ]:
#using the parquet
import json
import pandas as pd

df_loaded = pd.read_parquet('/content/drive/MyDrive/DSC 288 DS Grad Capstone/steam_tfidf_nn_recommender.parquet')

def show_competitors_from_parquet(df_in, game_name, top_n=50):
    row = df_in[df_in['name'].str.lower() == game_name.lower()]

    if row.empty:
        print(f'Game "{game_name}" not found.')
        return None

    comps = json.loads(row.iloc[0]['top_50_competitors_json'])
    comps_df = pd.DataFrame(comps).sort_values(by='similarity', ascending=False).head(top_n)

    print(f'\nTop {top_n} competitors for: {row.iloc[0]["name"]}\n')
    return comps_df.reset_index(drop=True)

display(show_competitors_from_parquet(df_loaded, 'Dota 2'))


Top 50 competitors for: Dota 2



,appid,name,similarity
0,572520,Dropzone,0.7327
1,339280,Strife®,0.6340
2,748940,Rise of Legions,0.6235
3,822710,Clash: Mutants Vs Pirates,0.6220
4,1467410,Isekai Eternal Alpha,0.6064
5,795580,THE DAY Online,0.5983
6,649770,Coregrounds,0.5916
7,1225790,Heat and Run,0.5755
8,703280,Zeal,0.5640
9,386360,SMITE®,0.5631


In [ ]:
suspect = 'Dropzone'

df_loaded.loc[
    df_loaded['name'].str.lower().isin(['dota 2', suspect.lower()]),
    ['appid', 'name', 'genres', 'tags']
]

,appid,name,genres,tags
2,570,Dota 2,"['Action', 'Strategy', 'Free To Play']","{'Free to Play': 59933, 'MOBA': 20158, 'Multip..."
7417,572520,Dropzone,"['Action', 'Free To Play', 'Strategy', 'Early ...","{'Free to Play': 69, 'Strategy': 68, 'MOBA': 6..."


In [ ]:
df_loaded.loc[
    df_loaded['name'].str.lower() == 'dota 2',
    ['appid', 'name', 'genres', 'tags']
]

,appid,name,genres,tags
2,570,Dota 2,"['Action', 'Strategy', 'Free To Play']","{'Free to Play': 59933, 'MOBA': 20158, 'Multip..."


In [ ]:
import ast
import pandas as pd

def parse_tag_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x == '':
            return {}
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, dict) else {}
        except:
            return {}
    return {}

def show_top_tags_from_loaded(df_in, game_name, top_n=20):
    row = df_in[df_in['name'].str.lower() == game_name.lower()]

    if row.empty:
        print(f'Game "{game_name}" not found.')
        return None

    tag_dict = parse_tag_dict(row.iloc[0]['tags'])

    if not tag_dict:
        print(f'No parsed tag dictionary found for "{game_name}".')
        return None

    out = pd.DataFrame(
        sorted(tag_dict.items(), key=lambda kv: kv[1], reverse=True)[:top_n],
        columns=['tag', 'count']
    )

    print(f'\nTop {top_n} tags for: {row.iloc[0]["name"]}\n')
    return out

display(show_top_tags_from_loaded(df_loaded, 'Dota 2', top_n=20))
display(show_top_tags_from_loaded(df_loaded, 'Dropzone', top_n=20))


Top 20 tags for: Dota 2



,tag,count
0,Free to Play,59933
1,MOBA,20158
2,Multiplayer,15359
3,Strategy,14252
4,e-sports,11780
5,Team-Based,10962
6,Competitive,8286
7,Action,7920
8,Online Co-Op,7464
9,PvP,6046



Top 20 tags for: Dropzone



,tag,count
0,Free to Play,69
1,Strategy,68
2,MOBA,66
3,Early Access,55
4,Action,50
5,PvP,49
6,Mechs,48
7,e-sports,40
8,RTS,36
9,Sci-fi,31


# EDA EXPERIMENTATION HERE
# PART 2(V2.0): Here is a quick showcase of our use of this parquet for DOTA 2

update: added date time data into the parquet, next let's update our original EDA notebook for the games.

In [4]:
#using the parquet
import json
import pandas as pd

df_loaded = pd.read_parquet('/content/drive/MyDrive/DSC 288 DS Grad Capstone/steam_tfidf_nn_recommender_v2.parquet')

def show_competitors(df_in, game_name, top_n=50, sort_by='similarity'):
    row = df_in[df_in['name'].str.lower() == game_name.lower()]

    if row.empty:
        print(f'Game "{game_name}" not found.')
        return None

    comps = row.iloc[0]['top_50_competitors']

    if isinstance(comps, str):
        comps = json.loads(comps)

    comps_df = pd.DataFrame(comps)

    if sort_by == 'release_date':
        comps_df['release_date'] = pd.to_datetime(comps_df['release_date'])
        comps_df = comps_df.sort_values('release_date', ascending=True)

    else:
        comps_df = comps_df.sort_values('similarity', ascending=False)

    print(f'\nTop {top_n} competitors for: {row.iloc[0]["name"]}\n')
    return comps_df.head(top_n).reset_index(drop=True)



In [6]:
show_competitors(df_rec, 'Dota 2')
show_competitors(df_rec, 'Dota 2', sort_by='similarity')


Top 50 competitors for: Dota 2


Top 50 competitors for: Dota 2



,appid,name,release_date,similarity
0,572520,Dropzone,2017-02-15,0.7327
1,339280,Strife®,2015-05-22,0.6340
2,748940,Rise of Legions,2019-08-30,0.6235
3,822710,Clash: Mutants Vs Pirates,2024-05-02,0.6220
4,1467410,Isekai Eternal Alpha,2021-07-14,0.6064
5,795580,THE DAY Online,2018-04-03,0.5983
6,649770,Coregrounds,2018-04-30,0.5916
7,1225790,Heat and Run,2022-10-13,0.5755
8,703280,Zeal,2018-09-24,0.5640
9,386360,SMITE®,2015-09-08,0.5631


In [7]:
def parse_tag_dict(x):
    if pd.isna(x):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if x == '':
            return {}
        try:
            parsed = ast.literal_eval(x)
            return parsed if isinstance(parsed, dict) else {}
        except:
            return {}
    return {}

def show_top_tags_from_loaded(df_in, game_name, top_n=20):
    row = df_in[df_in['name'].str.lower() == game_name.lower()]

    if row.empty:
        print(f'Game "{game_name}" not found.')
        return None

    tag_dict = parse_tag_dict(row.iloc[0]['tags'])

    if not tag_dict:
        print(f'No parsed tag dictionary found for "{game_name}".')
        return None

    out = pd.DataFrame(
        sorted(tag_dict.items(), key=lambda kv: kv[1], reverse=True)[:top_n],
        columns=['tag', 'count']
    )

    print(f'\nTop {top_n} tags for: {row.iloc[0]["name"]}\n')
    return out

display(show_top_tags_from_loaded(df_loaded, 'Dota 2', top_n=20))
display(show_top_tags_from_loaded(df_loaded, 'Dropzone', top_n=20))


Top 20 tags for: Dota 2



,tag,count
0,Free to Play,59933
1,MOBA,20158
2,Multiplayer,15359
3,Strategy,14252
4,e-sports,11780
5,Team-Based,10962
6,Competitive,8286
7,Action,7920
8,Online Co-Op,7464
9,PvP,6046



Top 20 tags for: Dropzone



,tag,count
0,Free to Play,69
1,Strategy,68
2,MOBA,66
3,Early Access,55
4,Action,50
5,PvP,49
6,Mechs,48
7,e-sports,40
8,RTS,36
9,Sci-fi,31
